# M3.2 The Displacement Field and Dielectric Boundary Value Problems

Book pointer: chapter 5, sections 5.3 (generalized Gauss' law), 5.4 (boundary conditions for E and D).

## 1. Recall

1. Define $\mathbf{D}$. Which charges are its sources? Write the generalized Gauss' law in both forms.
2. For a linear dielectric, relate $\mathbf{P}$, $\mathbf{D}$ and $\mathbf{E}$ through $\chi_e$ and $\epsilon_r$.
3. Derive the two boundary conditions at an interface: which component of $\mathbf{E}$ is continuous, which component of $\mathbf{D}$, and what breaks each one.
4. Combine $\mathbf{D} = \epsilon\mathbf{E}$, $\mathbf{E} = -\nabla V$ and $\nabla\cdot\mathbf{D} = \rho_f$ into one equation for $V$ when $\epsilon$ varies in space.
5. Parallel plates, half the gap filled with $\epsilon_r$: capacitance relative to the empty capacitor.

## 2. Principle to algorithm

With a linear dielectric the equation for the potential becomes $\nabla\cdot(\epsilon\nabla V) = -\rho_f$. On a grid the flux of $\mathbf{D}$ through the face between two nodes is $\epsilon_{\text{face}}(V_n - V_i)/h$ times the face area, and the sum of the six (or four) face fluxes equals the free charge in the cell. Which $\epsilon$ to put on a face matters at an interface: the harmonic mean of the two node values is the series combination of two half-cells, and it is the choice that keeps $D_n$ continuous. With it, the discrete interface sits halfway between the last node of one material and the first node of the other.

`di.solve_dielectric` assembles this as a sparse linear system in 2D or 3D. With $\epsilon_r = 1$ everywhere it is the Poisson solver from module 2. The fields follow from $\mathbf{E} = -\nabla V$ and $\mathbf{D} = \epsilon_r\epsilon_0\mathbf{E}$, and $\mathbf{P} = \mathbf{D} - \epsilon_0\mathbf{E}$.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax, dielectrics as di
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
# Parallel plates with a dielectric slab filling the left half of the gap.
n, eps_r, fill = 81, 4.0, 0.5
x = np.linspace(0, 1, n); h = x[1] - x[0]
slab = x < fill
k = np.argmax(~slab); x_int = 0.5 * (x[k - 1] + x[k])            # effective interface, see section 2
D_over_eps0 = 1.0 / (x_int / eps_r + (1 - x_int))
exact = np.where(x < x_int, D_over_eps0 * x / eps_r, D_over_eps0 * (x_int / eps_r + (x - x_int)))

V0 = np.zeros((n, n)); V0[:, 0] = V0[:, -1] = exact; V0[-1, :] = 1.0   # sides follow the 1D solution: no fringing
fixed = np.zeros((n, n), dtype=bool)
er = np.where(slab[:, None], eps_r, 1.0) * np.ones((1, n))

V = di.solve_dielectric(V0, fixed, er, h=h)
E = -fields.gradient(V, h, h)
D = EPS0 * er[..., None] * E
P = D - EPS0 * E

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
j = n // 2
axes[0].plot(x, V[:, j], label="solver"); axes[0].plot(x, exact, "k--", label="1D closed form"); axes[0].set_title("V(x)"); axes[0].legend()
axes[1].plot(x, E[:, j, 0] / E[:, j, 0].max(), label="E_x (scaled)"); axes[1].plot(x, D[:, j, 0] / D[:, j, 0].max(), label="D_x (scaled)")
axes[1].axvline(x_int, color="grey", ls=":"); axes[1].set_title("E jumps by eps_r, D is continuous"); axes[1].legend()
axes[2].plot(x, P[:, j, 0] / D[:, j, 0].max()); axes[2].set_title("P_x: only inside the dielectric")
for ax in axes: ax.axvspan(0, x_int, color="C0", alpha=0.08)
plt.show()
print(f"max |V - exact| = {np.abs(V - exact[:, None]).max():.1e}")
print(f"E_gap / E_slab = {E[k + 5, j, 0] / E[k - 6, j, 0]:.4f}   (eps_r = {eps_r})")

### Capacitance with a dielectric

Energy per unit depth is $\frac12\int\epsilon_r\epsilon_0 E^2\,dA$ and $C = 2U/V_0^2$. The closed form is two capacitors in series, one with the slab and one empty:

$$\frac{1}{C} = \frac{x_{\text{int}}}{\epsilon_r\epsilon_0} + \frac{1 - x_{\text{int}}}{\epsilon_0}.$$

How the energy is integrated matters here. A node-based integral assigns one permittivity to the interface node even though the central-difference $E$ there straddles both materials, which makes the energy first-order accurate in $h$. `di.energy` evaluates $E$ on the faces between nodes with the same harmonic-mean permittivity the solver uses, so it is consistent with the discrete model and reproduces the series formula to rounding error. Both are shown.

In [ ]:
def node_energy(V, er, h):
    E = -fields.gradient(V, h, h)
    u = 0.5 * EPS0 * er * np.sum(E**2, axis=-1)
    return float(np.trapezoid(np.trapezoid(u, dx=h, axis=1), dx=h, axis=0))

C_exact = 1 / (x_int / (eps_r * EPS0) + (1 - x_int) / EPS0)
C_node = 2 * node_energy(V, er, h)
C_face = 2 * di.energy(V, er, h)
print(f"series formula          {C_exact:.6e} F/m")
print(f"node-based energy       {C_node:.6e} F/m   ratio {C_node / C_exact:.5f}   (first order in h at the interface)")
print(f"face-based energy       {C_face:.6e} F/m   ratio {C_face / C_exact:.9f}")
print(f"gain over empty plates  {C_face / EPS0:.3f}")
assert abs(C_face / C_exact - 1) < 1e-9

### Exercise: a dielectric sphere in a uniform field (3D)

The closed form: inside the sphere the field is uniform, $\mathbf{E}_{\text{in}} = \dfrac{3}{\epsilon_r + 2}\mathbf{E}_0$, and outside the sphere adds a dipole term to the applied potential,

$$V_{\text{out}} = -E_0 z + \frac{\epsilon_r - 1}{\epsilon_r + 2}\,\frac{R^3 E_0 z}{r^3}.$$

Replace the body of `outer_boundary_potential` with your own implementation of $V_{\text{out}}$. The solver uses it on the faces of the box. The check compares the interior field to the closed form.

In [ ]:
eps_s, R, E0, m = 3.0, 1.0, 1.0, 61
L = 2.5 * R
xs, ys, zs, G = grids.grid3d((-L, L), (-L, L), (-L, L), m)
hs = grids.spacing(xs, ys, zs)[0]
r = grids.magnitude(G)

def outer_boundary_potential(G, r):
    kk = (eps_s - 1) / (eps_s + 2)
    with np.errstate(divide="ignore", invalid="ignore"):
        return -E0 * G[..., 2] + kk * R**3 * E0 * G[..., 2] / r**3       # replace with your own expression


V0s = np.where(r > R, outer_boundary_potential(G, r), 0.0)
ers = np.where(r <= R, eps_s, 1.0)
Vs = di.solve_dielectric(V0s, np.zeros(r.shape, dtype=bool), ers, h=hs)
Es = -fields.gradient(Vs, hs, hs, hs)
Ds = EPS0 * ers[..., None] * Es

deep = r < 0.6 * R
print(f"interior E_z: mean {Es[..., 2][deep].mean():.4f}, spread {Es[..., 2][deep].std():.4f}   closed form {3 / (eps_s + 2) * E0:.4f}")
assert abs(Es[..., 2][deep].mean() / (3 / (eps_s + 2) * E0) - 1) < 0.03

### Interactive 3D view

Equipotential shells of $V$ and the magnitude of $\mathbf{D}$, which is larger inside the sphere because $\mathbf{D} = \epsilon_r\epsilon_0\mathbf{E}$ while $E$ drops. Rotate and slice the figures.

In [ ]:
import plotly.graph_objects as go

# Every second grid point is plenty for the picture and keeps the figure data small.
sub = (slice(None, None, 2),) * 3
Xs, Ys, Zs = grids.components(G[sub])
Vsub = Vs[sub]
fig = go.Figure()
fig.add_trace(go.Isosurface(x=Xs.ravel(), y=Ys.ravel(), z=Zs.ravel(), value=Vsub.ravel(),
                            isomin=-1.5, isomax=1.5, surface_count=7, opacity=0.35, colorscale="RdBu", caps=dict(x_show=False, y_show=False, z_show=False),
                            colorbar=dict(title="V")))
th = np.linspace(0, 2 * np.pi, 60)
fig.add_trace(go.Scatter3d(x=R * np.cos(th), y=0 * th, z=R * np.sin(th), mode="lines", line=dict(color="black", width=4), name="sphere, y = 0"))
fig.update_layout(title="equipotential shells around a dielectric sphere", scene=dict(aspectmode="cube"), height=550, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

Dmag = grids.magnitude(Ds[sub]) / (EPS0 * E0)
fig2 = go.Figure(go.Volume(x=Xs.ravel(), y=Ys.ravel(), z=Zs.ravel(), value=Dmag.ravel(),
                           isomin=0.8, isomax=Dmag.max(), opacity=0.08, surface_count=12, colorscale="Viridis",
                           slices_y=dict(show=True, locations=[0.0]), caps=dict(x_show=False, y_show=False, z_show=False),
                           colorbar=dict(title="|D| / eps0 E0")))
fig2.update_layout(title="|D| is largest inside the sphere", scene=dict(aspectmode="cube"), height=550, margin=dict(l=0, r=0, t=40, b=0))
fig2.show()

## 4. Verify

The boundary conditions, measured. Walk out along radial lines through the sphere surface at several polar angles and record the tangential component of $\mathbf{E}$ and the normal component of $\mathbf{D}$ at $0.8R$ and $1.2R$. The two cannot be compared with each other directly, because the exterior field changes with $r$ between the sample points: the exact exterior tangential field is $E_0\sin\theta\,(1 - kR^3/r^3)$ with $k = (\epsilon_r-1)/(\epsilon_r+2)$, and the exact normal displacement is $\epsilon_0 E_0\cos\theta\,(1 + 2kR^3/r^3)$. So each sample is compared with the closed form at its own location. The closed forms are continuous at $r = R$, which you should verify by hand: inside, $E_t = E_{\text{in}}\sin\theta$ and $D_n = \epsilon_r\epsilon_0E_{\text{in}}\cos\theta$ with $E_{\text{in}} = 3E_0/(\epsilon_r+2)$.

Then compare the field on the $z$ axis to the closed forms. On the axis the normal component of $\mathbf{E}$ jumps by a factor $\epsilon_r$ at the pole. The staircase and the central difference smear that jump over about two cells, and the error there does not shrink with resolution, so the comparison excludes a band of $0.25R$ around the surface. Everywhere else the error is a few percent or less.

In [ ]:
from scipy.interpolate import RegularGridInterpolator
interpE = [RegularGridInterpolator((xs, ys, zs), Es[..., c]) for c in range(3)]
interpD = [RegularGridInterpolator((xs, ys, zs), Ds[..., c]) for c in range(3)]
def sample(interp, pts): return np.stack([f(pts) for f in interp], axis=-1)

kk = (eps_s - 1) / (eps_s + 2)
E_in_exact = 3 / (eps_s + 2) * E0
thetas = np.radians([20, 40, 60, 80, 100, 120, 140, 160])
nhat = np.stack([np.sin(thetas), 0 * thetas, np.cos(thetas)], axis=-1)
r_in, r_out = 0.8 * R, 1.2 * R

def tangential_and_normal(pts):
    E, D = sample(interpE, pts), sample(interpD, pts)
    Et = np.linalg.norm(E - np.sum(E * nhat, axis=1, keepdims=True) * nhat, axis=1)
    Dn = np.sum(D * nhat, axis=1)
    return Et, Dn

Et_in, Dn_in = tangential_and_normal(r_in * nhat)
Et_out, Dn_out = tangential_and_normal(r_out * nhat)
Et_in_exact = E_in_exact * np.sin(thetas)
Dn_in_exact = eps_s * EPS0 * E_in_exact * np.cos(thetas)
Et_out_exact = E0 * np.sin(thetas) * (1 - kk * R**3 / r_out**3)
Dn_out_exact = EPS0 * E0 * np.cos(thetas) * (1 + 2 * kk * R**3 / r_out**3)

print("numerical / closed form at each sample point")
print("theta   E_t inside   E_t outside   D_n inside   D_n outside")
for t, a, b, c_, d in zip(np.degrees(thetas), Et_in / Et_in_exact, Et_out / Et_out_exact, Dn_in / Dn_in_exact, Dn_out / Dn_out_exact):
    print(f"{t:5.0f}   {a:10.3f}   {b:11.3f}   {c_:10.3f}   {d:11.3f}")
ratios = np.concatenate([Et_in / Et_in_exact, Et_out / Et_out_exact, Dn_in / Dn_in_exact, Dn_out / Dn_out_exact])
print(f"largest deviation from the closed form: {np.abs(ratios - 1).max() * 100:.1f} %  (staircased sphere, h = {hs:.3f} R)")
assert np.abs(ratios - 1).max() < 0.08

In [ ]:
zl = np.linspace(-2.4, 2.4, 241)
Ez_num = interpE[2](np.stack([0 * zl, 0 * zl, zl], axis=-1))
kk = (eps_s - 1) / (eps_s + 2)
with np.errstate(divide="ignore"):
    Ez_exact = np.where(np.abs(zl) < R, 3 / (eps_s + 2) * E0, E0 + 2 * kk * R**3 * E0 / np.abs(zl) ** 3)
plt.plot(zl, Ez_exact, "k-", lw=3, alpha=0.3, label="closed form"); plt.plot(zl, Ez_num, "C1.", ms=3, label="solver")
plt.axvline(-R, color="grey", ls=":"); plt.axvline(R, color="grey", ls=":"); plt.xlabel("z / R"); plt.ylabel("E_z / E0"); plt.legend(); plt.show()
far = np.abs(np.abs(zl) - R) > 0.25
print(f"max relative error away from the surface: {np.abs(Ez_num / Ez_exact - 1)[far].max():.2e}")
assert np.abs(Ez_num / Ez_exact - 1)[far].max() < 0.03

## 5. Explore and rebuild

Explore. Repeat the sphere problem in 2D, which is a dielectric cylinder in a uniform transverse field. The closed form is $E_{\text{in}} = \dfrac{2}{\epsilon_r + 1}E_0$ with an exterior term $\dfrac{\epsilon_r - 1}{\epsilon_r + 1}\dfrac{R^2 E_0 x}{r^2}$. Use a 401-point grid and measure how the error in $E_{\text{in}}$ falls with grid spacing. Then replace the harmonic mean in a copy of the solver with the arithmetic mean and show that $D_n$ is no longer continuous.

Rebuild. Close this notebook and write the sparse assembly for $\nabla\cdot(\epsilon\nabla V) = -\rho_f$ from scratch in 2D, then verify it with the slab capacitor.